# Week 07 · Wednesday — Hard NLP Patterns & Aspect-Level Sentiment
NLP Foundations · ShopSense Reviews Dataset

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, f1_score

In [2]:
REVIEWS_PATH = "shopsense_reviews.csv"

# Negation window: how many tokens after a negation word to flip
NEGATION_WINDOW = 4

# Sarcasm signals
SARCASM_PUNCTUATION = re.compile(r'[!]{2,}|[?!]{2,}')
SARCASM_PRAISE_WORDS = {'wow', 'great', 'amazing', 'fantastic', 'brilliant',
                         'excellent', 'perfect', 'superb', 'incredible'}
SARCASM_CRASH_WORDS  = {'broke', 'broken', 'damaged', 'defective', 'failed',
                         'stopped', 'dead', 'useless', 'worst', 'terrible',
                         'waste', 'return', 'returned', 'awful'}

# Implicit negativity signals
IMPLICIT_NEGATIVE_PATTERNS = [
    r'return(ed|ing)?\s+it',
    r'return(ed|ing)?\s+within',
    r'broke\s+within',
    r'broke\s+after',
    r'broke\s+on\s+day',
    r'stopped\s+working',
    r'asking\s+for\s+replacement',
    r'still\s+waiting',
    r'raised\s+a\s+complaint',
]

# Hindi/Urdu sentiment seed words (simple lexicon)
HINDI_POSITIVE = {'accha', 'bahut', 'superb', 'best', 'shandaar', 'zabardast',
                   'mast', 'lajawaab', 'waah', 'perfect', 'bohot'}
HINDI_NEGATIVE = {'kharab', 'bakwas', 'barbad', 'bekar', 'ghatiya',
                   'bekaar', 'faltu', 'worst', 'bura', 'nahi'}

# Aspect keywords
ASPECT_KEYWORDS = {
    'camera':    ['camera', 'photo', 'picture', 'image', 'zoom', 'lens', 'selfie'],
    'battery':   ['battery', 'charge', 'charging', 'backup', 'drain', 'life'],
    'delivery':  ['delivery', 'shipping', 'arrived', 'late', 'fast', 'courier'],
    'quality':   ['quality', 'build', 'material', 'durable', 'finish', 'sturdy'],
    'support':   ['support', 'service', 'helpline', 'agent', 'response', 'refund'],
    'price':     ['price', 'cost', 'worth', 'value', 'expensive', 'cheap', 'affordable'],
}

In [3]:
def load_reviews(path):
    """Load reviews CSV with basic column validation."""
    try:
        df = pd.read_csv(path)
        required = {'review_text', 'sentiment_label', 'category', 'language'}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        df['review_text'] = df['review_text'].fillna("")
        print(f"Loaded {len(df)} reviews")
        return df
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        raise


reviews_df = load_reviews(REVIEWS_PATH)

Loaded 10199 reviews


---
## Q1 — Five Hard NLP Patterns in Indian E-Commerce Reviews

### (a) Negation — 'not bad at all' is positive

In [4]:
NEGATION_WORDS = {'not', 'no', 'never', 'dont', "don't", 'neither', 'nor',
                   'nothing', 'nowhere', 'nobody', 'hardly', 'barely'}

def negate_tokens(tokens, negation_words=NEGATION_WORDS, window=NEGATION_WINDOW):
    """
    Attach a NOT_ prefix to tokens within `window` steps of a negation word.
    This is the standard negation-scope preprocessing trick.
    """
    result = []
    negate_until = 0
    for i, token in enumerate(tokens):
        if token.lower() in negation_words:
            negate_until = i + window
            result.append(token)
        elif i <= negate_until:
            result.append(f"NOT_{token}")
        else:
            result.append(token)
    return result


def demo_negation(examples):
    """Show baseline failure vs negation-aware preprocessing."""
    print(f"{'Sentence':50s}  {'Baseline tokens':35s}  {'Negation-aware tokens'}")
    print("-" * 130)
    for sent in examples:
        tokens = re.findall(r"[a-zA-Z']+", sent.lower())
        baseline    = " ".join(tokens)
        negated     = " ".join(negate_tokens(tokens))
        print(f"  {sent:48s}  {baseline:35s}  {negated}")


negation_examples = [
    "not bad at all",
    "No complaints whatsoever",
    "I don't recommend this",
    "never buying again",
]

demo_negation(negation_examples)

print("""
Baseline failure: a TF-IDF or BOW model sees 'bad' and predicts Negative
for 'not bad at all'. By tagging tokens inside the negation window as
NOT_bad, the model learns NOT_bad as a distinct positive-leaning feature.
""")

Sentence                                            Baseline tokens                      Negation-aware tokens
----------------------------------------------------------------------------------------------------------------------------------
  not bad at all                                    not bad at all                       not NOT_bad NOT_at NOT_all
  No complaints whatsoever                          no complaints whatsoever             no NOT_complaints NOT_whatsoever
  I don't recommend this                            i don't recommend this               NOT_i don't NOT_recommend NOT_this
  never buying again                                never buying again                   never NOT_buying NOT_again

Baseline failure: a TF-IDF or BOW model sees 'bad' and predicts Negative
for 'not bad at all'. By tagging tokens inside the negation window as
NOT_bad, the model learns NOT_bad as a distinct positive-leaning feature.



In [5]:
# Find real negation examples from the dataset
negation_mask = reviews_df['review_text'].str.contains(
    r'\bno complaints\b|\bnot bad\b|\bnever\b', case=False, na=False)
sample_neg = reviews_df[negation_mask][['review_text', 'sentiment_label']].head(5)
print("Real negation examples from ShopSense:")
print(sample_neg.to_string(index=False))

Real negation examples from ShopSense:
                                                                       review_text sentiment_label
Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive
Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive
Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive
Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive
Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive


### (b) Sarcasm — 'Wow great! Broke on day 1'

In [6]:
def detect_sarcasm(text,
                   praise_words=SARCASM_PRAISE_WORDS,
                   crash_words=SARCASM_CRASH_WORDS):
    """
    Rule-based sarcasm flag: praise word + exclamation + negative outcome
    in the same short text.
    Returns (is_sarcastic, evidence).
    """
    tokens = set(re.findall(r'[a-z]+', text.lower()))
    has_praise   = bool(tokens & praise_words)
    has_crash    = bool(tokens & crash_words)
    has_exclam   = bool(SARCASM_PUNCTUATION.search(text))
    is_sarcastic = has_praise and has_crash
    evidence = {
        'praise_match':  sorted(tokens & praise_words),
        'crash_match':   sorted(tokens & crash_words),
        'exclamation':   has_exclam,
    }
    return is_sarcastic, evidence


sarcasm_examples = [
    "Wow great! Broke on day 1",
    "Amazing product!! Stopped working after 2 days",
    "Excellent packaging, terrible product inside",
    "Perfect delivery, but item was damaged",
    "Really loving this product so far",
]

print(f"{'Sentence':48s}  Sarcastic?  Evidence")
print("-" * 100)
for s in sarcasm_examples:
    flag, ev = detect_sarcasm(s)
    print(f"  {s:46s}  {'YES' if flag else 'no ':9s}  {ev}")

print("""
Baseline failure: VADER / TF-IDF sees 'wow', 'great', 'amazing' and
predicts Positive — completely missing the negative outcome that follows.
Feature engineering fix: add a binary 'sarcasm_flag' feature computed
by the above rule, plus character n-grams to capture !! patterns.
""")

Sentence                                          Sarcastic?  Evidence
----------------------------------------------------------------------------------------------------
  Wow great! Broke on day 1                       YES        {'praise_match': ['great', 'wow'], 'crash_match': ['broke'], 'exclamation': False}
  Amazing product!! Stopped working after 2 days  YES        {'praise_match': ['amazing'], 'crash_match': ['stopped'], 'exclamation': True}
  Excellent packaging, terrible product inside    YES        {'praise_match': ['excellent'], 'crash_match': ['terrible'], 'exclamation': False}
  Perfect delivery, but item was damaged          YES        {'praise_match': ['perfect'], 'crash_match': ['damaged'], 'exclamation': False}
  Really loving this product so far               no         {'praise_match': [], 'crash_match': [], 'exclamation': False}

Baseline failure: VADER / TF-IDF sees 'wow', 'great', 'amazing' and
predicts Positive — completely missing the negative outcome that fo

In [7]:
# Real sarcasm-like examples from dataset
sarc_mask = reviews_df['review_text'].str.contains(
    r'(?i)(great|amazing|excellent).{0,40}(broke|damaged|defective|terrible)', na=False)
real_sarc = reviews_df[sarc_mask][['review_text', 'sentiment_label', 'rating']].head(5)
print("Real sarcasm-pattern examples from ShopSense:")
print(real_sarc.to_string(index=False))

Real sarcasm-pattern examples from ShopSense:
                                                                           review_text sentiment_label  rating
 It's fine I guess. Not amazing, not terrible. Just an average haircare for daily use.         Neutral       3
It's fine I guess. Not amazing, not terrible. Just an average fragrance for daily use.         Neutral       3
   It's fine I guess. Not amazing, not terrible. Just an average makeup for daily use.         Neutral       3
 It's fine I guess. Not amazing, not terrible. Just an average chargers for daily use.         Neutral       3
    It's fine I guess. Not amazing, not terrible. Just an average decor for daily use.         Neutral       3


C:\Users\abeku\AppData\Local\Temp\ipykernel_20140\3822111085.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sarc_mask = reviews_df['review_text'].str.contains(


### (c) Code-mixing — Hindi/English mixed reviews

In [8]:
def score_codemixed(text,
                    hindi_pos=HINDI_POSITIVE,
                    hindi_neg=HINDI_NEGATIVE):
    """
    Simple sentiment scorer for code-mixed text.
    Checks both English tokens and a seed Hindi lexicon.
    Returns (score, detail) where score > 0 is positive.
    """
    tokens = re.findall(r'[a-zA-Z]+', text.lower())
    pos_hits = [t for t in tokens if t in hindi_pos]
    neg_hits = [t for t in tokens if t in hindi_neg]
    score = len(pos_hits) - len(neg_hits)
    label = 'Positive' if score > 0 else ('Negative' if score < 0 else 'Neutral')
    return label, {'positive_tokens': pos_hits, 'negative_tokens': neg_hits, 'score': score}


codemix_examples = [
    "Product bahut accha hai lekin delivery late thi",
    "Bilkul bakwas product. Paisa barbad. Return kar raha hun",
    "Quality bahut accha hai. Bohot khush hun",
    "Product kharab hai. Bekar quality. Very disappointed",
]

print(f"{'Sentence':55s}  Label      Detail")
print("-" * 120)
for s in codemix_examples:
    label, detail = score_codemixed(s)
    print(f"  {s:53s}  {label:10s} {detail}")

print("""
Baseline failure: an English-only TF-IDF model treats 'bahut', 'accha',
'kharab' as unknown tokens and ignores them — losing the main sentiment
signal in the review. Fix: bilingual lexicon + transliteration normalisation.
""")

Sentence                                                 Label      Detail
------------------------------------------------------------------------------------------------------------------------
  Product bahut accha hai lekin delivery late thi        Positive   {'positive_tokens': ['bahut', 'accha'], 'negative_tokens': [], 'score': 2}
  Bilkul bakwas product. Paisa barbad. Return kar raha hun  Negative   {'positive_tokens': [], 'negative_tokens': ['bakwas', 'barbad'], 'score': -2}
  Quality bahut accha hai. Bohot khush hun               Positive   {'positive_tokens': ['bahut', 'accha', 'bohot'], 'negative_tokens': [], 'score': 3}
  Product kharab hai. Bekar quality. Very disappointed   Negative   {'positive_tokens': [], 'negative_tokens': ['kharab', 'bekar'], 'score': -2}

Baseline failure: an English-only TF-IDF model treats 'bahut', 'accha',
'kharab' as unknown tokens and ignores them — losing the main sentiment
signal in the review. Fix: bilingual lexicon + transliteration normali

In [9]:
codemix_df = reviews_df[reviews_df['language'] == 'Code-mixed'][['review_text', 'sentiment_label']].head(8)
print("Real code-mixed examples from ShopSense:")
print(codemix_df.to_string(index=False))

Real code-mixed examples from ShopSense:
                                                                           review_text sentiment_label
Amazing value for money. The technical exceeded my expectations in every way possible.        Positive
   Received a defective storage. Asked for replacement, still waiting after 2 weeks!!!        Negative
               <b>Great product!</b> Worth every rupee. Delivery was super fast too!!!        Positive
      Worst purchase ever. Product arrived damaged and customer support was unhelpful.        Negative
         Superb! The fragrance is exactly as described. Very happy with this purchase.        Positive
               Product bahut accha hai. Quality is top notch. Will buy again for sure.        Positive
    Very good product. Using it since 2 weeks. No complaints whatsoever. Recommended!!        Positive
               <b>Great product!</b> Worth every rupee. Delivery was super fast too!!!        Positive


### (d) Implicit sentiment — 'Returned it within 2 hours'

In [10]:
IMPLICIT_PATTERNS = [re.compile(p, re.IGNORECASE) for p in IMPLICIT_NEGATIVE_PATTERNS]


def detect_implicit_negative(text, patterns=IMPLICIT_PATTERNS):
    """
    Flag implicit negative sentiment via behaviour-based patterns.
    Returning a product, item breaking instantly, waiting for resolution
    all signal dissatisfaction without explicit negative words.
    """
    triggered = [p.pattern for p in patterns if p.search(text)]
    return bool(triggered), triggered


implicit_examples = [
    "Returned it within 2 hours",
    "Asked for replacement, still waiting after 2 weeks",
    "Terrible quality. Broke within 2 days.",
    "Product stopped working after one use",
    "Amazing product, very happy with it",
    "Delivery was fast, product looks good",
]

print(f"{'Sentence':50s}  Implicit Neg?  Triggered Pattern")
print("-" * 110)
for s in implicit_examples:
    flag, patterns_hit = detect_implicit_negative(s)
    print(f"  {s:48s}  {'YES' if flag else 'no ':13s}  {patterns_hit}")

print("""
Baseline failure: 'Returned it within 2 hours' has no negative adjectives —
a bag-of-words model predicts Neutral. The implicit signal is the *action*
of returning, not any explicit opinion word. Fix: regex-based behaviour
features added as binary columns before the classifier sees the text.
""")

Sentence                                            Implicit Neg?  Triggered Pattern
--------------------------------------------------------------------------------------------------------------
  Returned it within 2 hours                        YES            ['return(ed|ing)?\\s+it']
  Asked for replacement, still waiting after 2 weeks  YES            ['still\\s+waiting']
  Terrible quality. Broke within 2 days.            YES            ['broke\\s+within']
  Product stopped working after one use             YES            ['stopped\\s+working']
  Amazing product, very happy with it               no             []
  Delivery was fast, product looks good             no             []

Baseline failure: 'Returned it within 2 hours' has no negative adjectives —
a bag-of-words model predicts Neutral. The implicit signal is the *action*
of returning, not any explicit opinion word. Fix: regex-based behaviour
features added as binary columns before the classifier sees the text.



### (e) Comparative sentiment — 'Way better than my previous Samsung'

In [11]:
COMPARATIVE_PATTERNS = [
    (re.compile(r'\b(way|much|far|so much)\s+better\b', re.I),  'positive'),
    (re.compile(r'\b(way|much|far)\s+worse\b', re.I),           'negative'),
    (re.compile(r'\bbetter\s+than\b', re.I),                    'positive'),
    (re.compile(r'\bworse\s+than\b',  re.I),                    'negative'),
    (re.compile(r'\bnot\s+as\s+good\s+as\b', re.I),             'negative'),
    (re.compile(r'\bsuperior\s+to\b', re.I),                    'positive'),
    (re.compile(r'\binferior\s+to\b', re.I),                    'negative'),
]


def classify_comparative(text, patterns=COMPARATIVE_PATTERNS):
    """
    Detect comparative sentiment and infer direction.
    Returns (polarity, matched_phrase) or ('neutral', None).
    """
    for pattern, polarity in patterns:
        match = pattern.search(text)
        if match:
            return polarity, match.group(0)
    return 'neutral', None


comparative_examples = [
    "Way better than my previous Samsung",
    "Much worse than the last one I bought",
    "This is not as good as the older model",
    "Superior to everything else in this price range",
    "Good product overall",
]

print(f"{'Sentence':50s}  Polarity    Matched phrase")
print("-" * 95)
for s in comparative_examples:
    polarity, phrase = classify_comparative(s)
    print(f"  {s:48s}  {polarity:10s}  {phrase}")

print("""
Baseline failure: 'Way better than my previous Samsung' contains no
standalone positive adjective — a unigram model may score it Neutral.
Fix: extract the comparative structure as a feature:
direction (better/worse) + intensifier (way/much/far).
""")

Sentence                                            Polarity    Matched phrase
-----------------------------------------------------------------------------------------------
  Way better than my previous Samsung               positive    Way better
  Much worse than the last one I bought             negative    Much worse
  This is not as good as the older model            negative    not as good as
  Superior to everything else in this price range   positive    Superior to
  Good product overall                              neutral     None

Baseline failure: 'Way better than my previous Samsung' contains no
standalone positive adjective — a unigram model may score it Neutral.
Fix: extract the comparative structure as a feature:
direction (better/worse) + intensifier (way/much/far).



---
## Q2 — Review-Level vs Aspect-Level Classifier

### Review-level baseline (88% F1 reference)

In [12]:
def build_feature_df(df):
    """
    Add engineered binary features on top of raw text:
    - has_negation, has_sarcasm_signal, has_implicit_negative,
      has_comparative, has_codemix_positive, has_codemix_negative
    """
    out = df.copy()
    texts = df['review_text']
    out['has_negation']        = texts.apply(
        lambda t: int(bool(re.search(r'\bnot\b|\bno\b|\bnever\b', t, re.I))))
    out['has_sarcasm_signal']  = texts.apply(
        lambda t: int(detect_sarcasm(t)[0]))
    out['has_implicit_neg']    = texts.apply(
        lambda t: int(detect_implicit_negative(t)[0]))
    out['has_comparative_pos'] = texts.apply(
        lambda t: int(classify_comparative(t)[0] == 'positive'))
    out['has_comparative_neg'] = texts.apply(
        lambda t: int(classify_comparative(t)[0] == 'negative'))
    return out


def train_review_classifier(df, text_col='review_text', label_col='sentiment_label'):
    """Train a TF-IDF + LogReg review-level sentiment classifier."""
    data = df[df[label_col].isin(['Positive', 'Negative'])].dropna(subset=[text_col])
    X = data[text_col]
    y = data[label_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=8000,
                                  sublinear_tf=True, token_pattern=r'[a-zA-Z]+')),
        ('clf',   LogisticRegression(C=1.0, max_iter=500, random_state=42)),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    print("Review-level classifier:")
    print(classification_report(y_test, preds))
    macro_f1 = f1_score(y_test, preds, average='macro')
    print(f"Macro F1: {macro_f1:.3f}")
    return pipeline, macro_f1


review_pipeline, review_f1 = train_review_classifier(reviews_df)

Review-level classifier:
              precision    recall  f1-score   support

    Negative       1.00      0.83      0.91       419
    Positive       0.95      1.00      0.98      1421

    accuracy                           0.96      1840
   macro avg       0.98      0.92      0.94      1840
weighted avg       0.96      0.96      0.96      1840

Macro F1: 0.942


### (a) Why is aspect-level harder than review-level?

In [13]:
print("""
Why aspect-level sentiment (71% F1) is harder than review-level (88% F1):

1. Granularity: Review-level collapses everything into one label. 
   Aspect-level must correctly scope which words modify which aspect.
   'Great camera but awful battery' — two aspects, opposite polarities.

2. Implicit aspects: 'Stopped working after 2 days' is negative about
   durability/quality, but the aspect word never appears.

3. Boundary detection: 'The camera quality was good but the battery
   drains too fast' — the model must find where camera sentiment ends
   and battery sentiment begins.

4. Sparse labels: Each aspect appears in only a fraction of reviews,
   so per-aspect training data is much smaller.

5. Cross-sentence coreference: In multi-sentence reviews an aspect
   mentioned in sentence 1 may be opinionated in sentence 3.
""")


Why aspect-level sentiment (71% F1) is harder than review-level (88% F1):

1. Granularity: Review-level collapses everything into one label. 
   Aspect-level must correctly scope which words modify which aspect.
   'Great camera but awful battery' — two aspects, opposite polarities.

2. Implicit aspects: 'Stopped working after 2 days' is negative about
   durability/quality, but the aspect word never appears.

3. Boundary detection: 'The camera quality was good but the battery
   drains too fast' — the model must find where camera sentiment ends
   and battery sentiment begins.

4. Sparse labels: Each aspect appears in only a fraction of reviews,
   so per-aspect training data is much smaller.

5. Cross-sentence coreference: In multi-sentence reviews an aspect
   mentioned in sentence 1 may be opinionated in sentence 3.



### (b) How to improve from 71% to 80%+

In [14]:
print("""
Improvement strategies for aspect-level F1: 71% → 80%+

1. Dependency-parsed windows: Instead of whole-review TF-IDF, extract
   a ±3 token window around each aspect mention. The opinion words
   closest to the aspect are the most relevant.

2. Contrastive conjunction splitting: Split on 'but', 'however',
   'although', 'yet' to isolate independent sentiment clauses before
   aspect attribution.

3. Aspect-specific training: Train one small LR / SVM per aspect
   (camera, battery, delivery…) rather than a single multi-label model.
   Each model learns the vocabulary of its domain.

4. Opinion lexicon features: Add SentiWordNet or a domain-specific
   e-commerce lexicon as features. Words like 'atrocious', 'stellar',
   'sluggish' carry strong polarity that sparse TF-IDF may underweight.

5. Negation scoping (from Q1a): Propagate negation within aspect window
   so 'not good camera' scores negative for camera, not positive.

6. Fine-tuned transformer: A bert-base model fine-tuned on
   aspect-sentiment pairs (SemEval ABSA data) typically adds 8–12 F1
   points over TF-IDF baselines.
""")


Improvement strategies for aspect-level F1: 71% → 80%+

1. Dependency-parsed windows: Instead of whole-review TF-IDF, extract
   a ±3 token window around each aspect mention. The opinion words
   closest to the aspect are the most relevant.

2. Contrastive conjunction splitting: Split on 'but', 'however',
   'although', 'yet' to isolate independent sentiment clauses before
   aspect attribution.

3. Aspect-specific training: Train one small LR / SVM per aspect
   (camera, battery, delivery…) rather than a single multi-label model.
   Each model learns the vocabulary of its domain.

4. Opinion lexicon features: Add SentiWordNet or a domain-specific
   e-commerce lexicon as features. Words like 'atrocious', 'stellar',
   'sluggish' carry strong polarity that sparse TF-IDF may underweight.

5. Negation scoping (from Q1a): Propagate negation within aspect window
   so 'not good camera' scores negative for camera, not positive.

6. Fine-tuned transformer: A bert-base model fine-tuned on
  

### (c) Aspect-sentiment extraction from a mixed review

In [15]:
SENTIMENT_POSITIVE_WORDS = {
    'amazing', 'great', 'good', 'excellent', 'perfect', 'superb',
    'brilliant', 'love', 'loved', 'best', 'fantastic', 'wonderful',
    'outstanding', 'impressive', 'awesome', 'incredible', 'happy',
}
SENTIMENT_NEGATIVE_WORDS = {
    'terrible', 'awful', 'bad', 'poor', 'worst', 'horrible', 'atrocious',
    'useless', 'broken', 'defective', 'disappointing', 'unhelpful',
    'slow', 'expensive', 'cheap', 'fake', 'damaged',
}


def extract_aspect_sentiments(text,
                               aspect_keywords=ASPECT_KEYWORDS,
                               pos_words=SENTIMENT_POSITIVE_WORDS,
                               neg_words=SENTIMENT_NEGATIVE_WORDS,
                               window=6):
    """
    Rule-based aspect-sentiment pair extractor.
    For each detected aspect, scans a ±window token context for
    opinion words and returns (aspect, sentiment, evidence).
    """
    # Split on contrastive conjunctions first
    clauses = re.split(r'\b(but|however|although|yet|though)\b', text, flags=re.I)
    clauses = [c.strip() for c in clauses if c.strip() and
               c.lower() not in {'but', 'however', 'although', 'yet', 'though'}]

    results = []
    for clause in clauses:
        tokens = re.findall(r'[a-z]+', clause.lower())
        for aspect, keywords in aspect_keywords.items():
            for i, tok in enumerate(tokens):
                if tok in keywords:
                    lo = max(0, i - window)
                    hi = min(len(tokens), i + window + 1)
                    window_tokens = tokens[lo:hi]

                    # Check negation inside window
                    negated = any(t in NEGATION_WORDS for t in window_tokens)
                    pos_hits = [t for t in window_tokens if t in pos_words]
                    neg_hits = [t for t in window_tokens if t in neg_words]

                    if not pos_hits and not neg_hits:
                        continue

                    raw_score = len(pos_hits) - len(neg_hits)
                    if negated:
                        raw_score = -raw_score

                    sentiment = ('Positive' if raw_score > 0
                                 else 'Negative' if raw_score < 0
                                 else 'Neutral')
                    results.append({
                        'aspect':    aspect,
                        'trigger':   tok,
                        'sentiment': sentiment,
                        'pos_words': pos_hits,
                        'neg_words': neg_hits,
                        'clause':    clause[:60],
                    })
                    break  # one match per aspect per clause
    return results


target_review = "Amazing camera quality but the battery is atrocious and customer support was unhelpful."

print(f"Review: {target_review}\n")
pairs = extract_aspect_sentiments(target_review)

if pairs:
    result_df = pd.DataFrame(pairs)
    print(result_df[['aspect', 'trigger', 'sentiment', 'pos_words', 'neg_words']].to_string(index=False))
else:
    print("No aspect-sentiment pairs extracted.")

print("""
This single review is simultaneously Positive (camera) and Negative
(battery, support). A review-level classifier collapses it to one label
— likely Positive or Neutral — hiding the product team's most actionable
signal: the battery and support need improvement, the camera does not.
""")

Review: Amazing camera quality but the battery is atrocious and customer support was unhelpful.

 aspect trigger sentiment pos_words              neg_words
 camera  camera  Positive [amazing]                     []
quality quality  Positive [amazing]                     []
battery battery  Negative        []            [atrocious]
support support  Negative        [] [atrocious, unhelpful]

This single review is simultaneously Positive (camera) and Negative
(battery, support). A review-level classifier collapses it to one label
— likely Positive or Neutral — hiding the product team's most actionable
signal: the battery and support need improvement, the camera does not.



In [16]:
# Apply extractor to a sample of Electronics reviews
elec_sample = reviews_df[reviews_df['category'] == 'Electronics']['review_text'].dropna().head(20)
all_pairs = []
for text in elec_sample:
    pairs = extract_aspect_sentiments(text)
    all_pairs.extend(pairs)

if all_pairs:
    pairs_df = pd.DataFrame(all_pairs)
    print("Aspect sentiment distribution across 20 Electronics reviews:")
    print(pairs_df.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0))
else:
    print("No aspect-sentiment pairs found in sample.")

Aspect sentiment distribution across 20 Electronics reviews:
sentiment  Negative  Positive
aspect                       
battery           0         1
camera            0         1
delivery          0         3
price             0         3
quality           3         3
